In [ ]:
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent.parent / 'src' / 'display_html.py').read())

## Schapiro et al. (2017) — Full Figure Reproduction

This notebook reproduces all three main result figures from:

> Schapiro, A. C., Turk-Browne, N. B., Botvinick, M. M., & Norman, K. A. (2017). Complementary learning systems within the hippocampus. *Phil. Trans. R. Soc. B*, 372, 20160049.

**Testing procedure (§2.d):** present each item in isolation — no previous item, no plus-phase target — and let activity settle for 80 cycles. Record at cycle 20 (initial response) and cycle 80 (settled response).

**Three experiments:**
- **Fig 2** — Pair structure: 8 items in 4 pairs (AB/CD/EF/GH); with vs without statistical learning
- **Fig 3** — Community structure: 15 items in 5 communities × 3 items; MSP/TSP dissociation
- **Fig 4** — Associative inference: 9 items in 3 triads; CA3 recurrence enables transitivity

Note: the paper averages over 500 network seeds. We use `N_REPS` (default 20) for reasonable run time.

In [ ]:
import sys
from pathlib import Path

DIR_SRC = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
DIR_VIZ = (Path(__vsc_ipynb_file__).parent.parent / 'visualizations').resolve()
DIR_MAN = (Path(__vsc_ipynb_file__).parent.parent / 'manuscript').resolve()
sys.path.insert(0, DIR_SRC)

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from model import M_Hip
from tasks import T_PairDataset, T_CommunityGraphDataset, T_ChainDataset
from simulate import run_simulation
from util import F_pearson_sim_mat, F_extract_rsa, F_pattern_sim_by_mask
from visualize import plot_rsa_heatmap, plot_sim_bars, plot_output_prob

# Number of network initializations to average over.
# Schapiro (2017) §2.a.v: 500. We use 20 for practical run time.
N_REPS = 20

print('Setup complete.')

In [ ]:
# Testing procedure (Schapiro 2017 §2.d): model.run_evaluation(item_idx) and
# model.run_evaluation_all() are M_Hip methods defined in model.py.
# Analysis helpers: F_pearson_sim_mat, F_pattern_sim_by_mask imported from util.py.
print('Analysis helpers ready.')

## Figure 2 — Pair Structure

Schapiro (2017) §3.a: 8 items in 4 fixed pairs (AB/CD/EF/GH).

**Two conditions** (Fig 2 top and bottom rows):
- **Episodic** (`interleaved=True`): pairs are presented in isolation — A→B, C→D, etc. — with no B→A or cross-pair links. Only TSP can learn (direct pair associations). This is the "without statistical learning" condition.
- **Statistical** (`interleaved=False`): sequential walk — A→B (deterministic), B→A_other (uniform over 3 remaining pairs). Both MSP and TSP learn. MSP detects the forward asymmetry; TSP binds individual A→B episodes.

**Metrics:**
- RSA matrices: 8×8 pairwise Pearson r between CA1 representations (initial and settled)
- Pattern similarity: mean Pearson r for same-pair items vs cross-pair (shuffled baseline)
- Output probability: P(ECout_partner > 0.5) after each epoch — learning curve over 10 epochs

**Expected results (Schapiro 2017 Fig. 2):**
- Both conditions produce CA1 similarity above baseline for same-pair items
- Statistical condition shows smoother community-like RSA structure in CA1 (MSP contribution)
- Episodic condition shows sharper pair-specific patterns (TSP dominates)
- Output probability increases over epochs, faster for episodic (TSP: fast lr=0.4)

In [ ]:
# --- Fig 2: Pair structure training + testing ---
# 8 items (n_cond=8), 4 pairs (AB/CD/EF/GH)
# 80 trials/epoch × 10 epochs (Schapiro 2017 §3.a)
# Two conditions × N_REPS seeds each

N_ITEMS_PAIR  = 8
N_EPOCHS_PAIR = 10
N_TRIALS_PAIR = 80
N_PAIRS       = 4

# Template instance: not used for data, only for structural methods (rsa_masks, output_prob_by_epoch)
ds_pair = T_PairDataset(n_steps=N_TRIALS_PAIR, n_pairs=N_PAIRS)

# eval_fn returns settled representations (§2.d: 80 cycles settling per item)
# result['eval'][r][ep+1] = settled_mats after epoch ep for rep r
# result['eval'][r][0]    = settled_mats before training (not used here)

print('Running episodic condition...')
ep_results = run_simulation(
    lambda: M_Hip(n_cond=N_ITEMS_PAIR),
    lambda: T_PairDataset(n_steps=N_TRIALS_PAIR, n_pairs=N_PAIRS, interleaved=True),
    n_epochs=N_EPOCHS_PAIR, n_reps=N_REPS,
    eval_fn=lambda m: m.run_evaluation_all()[1],
    seed=0,
)

print('Running statistical condition...')
sl_results = run_simulation(
    lambda: M_Hip(n_cond=N_ITEMS_PAIR),
    lambda: T_PairDataset(n_steps=N_TRIALS_PAIR, n_pairs=N_PAIRS, interleaved=False),
    n_epochs=N_EPOCHS_PAIR, n_reps=N_REPS,
    eval_fn=lambda m: m.run_evaluation_all()[1],
    seed=0,
)

# ECout activity per rep per epoch — used for output probability curves
ep_ecout = np.stack([
    np.stack([ep_results['eval'][r][ep + 1]['ecout'] for ep in range(N_EPOCHS_PAIR)])
    for r in range(N_REPS)
])  # (N_REPS, N_EPOCHS_PAIR, N_ITEMS_PAIR, N_ITEMS_PAIR)
sl_ecout = np.stack([
    np.stack([sl_results['eval'][r][ep + 1]['ecout'] for ep in range(N_EPOCHS_PAIR)])
    for r in range(N_REPS)
])

# Pearson r RSA matrices from final-epoch settled representations
ep_rsa = F_extract_rsa(ep_results, lambda e: e[-1])
sl_rsa = F_extract_rsa(sl_results, lambda e: e[-1])

print('Done.')

In [ ]:
# --- Fig 2 metrics ---
epochs  = np.arange(1, N_EPOCHS_PAIR + 1)
ep_prob = ds_pair.output_prob_by_epoch(ep_ecout)
sl_prob = ds_pair.output_prob_by_epoch(sl_ecout)
masks   = ds_pair.rsa_masks()
ep_sim  = F_pattern_sim_by_mask(ep_rsa, masks)
sl_sim  = F_pattern_sim_by_mask(sl_rsa, masks)

for cond, sim in [('Episodic', ep_sim), ('Statistical', sl_sim)]:
    print(f'{cond}:')
    for l in ['dg', 'ca3', 'ca1']:
        s, c = sim[l]['same'][0], sim[l]['cross'][0]
        print(f'  {l.upper():4s}: same={s:.3f}  cross={c:.3f}  diff={s-c:.3f}')

In [ ]:
# --- Plot Figure 2 ---
fig = plt.figure(figsize=(14, 7))
gs  = gridspec.GridSpec(2, 6, figure=fig, hspace=0.45, wspace=0.5)
layers     = ['dg', 'ca3', 'ca1']
conditions = [('Episodic (without SL)', ep_rsa, ep_sim, ep_prob),
              ('Statistical (with SL)',  sl_rsa, sl_sim, sl_prob)]

for row, (cond_name, rsa, sim, prob) in enumerate(conditions):
    for col, layer in enumerate(layers):
        t = f'RSA (settled)\n{layer.upper()}' if (row == 0 and col == 0) else layer.upper()
        plot_rsa_heatmap(fig.add_subplot(gs[row, col]), rsa[layer].mean(0),
                         N_PAIRS, 2, title=t,
                         ylabel=cond_name if col == 0 else None)

    ax = fig.add_subplot(gs[row, 3])
    plot_sim_bars(ax, sim, layers, ['same', 'cross'], colors=['C0', 'C1'])
    ax.set_ylim(-0.3, 1.1)

    plot_output_prob(fig.add_subplot(gs[row, 4:]), epochs,
                     {'correct partner': prob}, chance=2 / N_ITEMS_PAIR,
                     title='Output probability\n(correct partner)')

fig.suptitle('Figure 2 — Pair Structure (Schapiro 2017)', fontsize=11, fontweight='bold')
plt.savefig(DIR_VIZ / 'Schapiro2017_Fig2_pair.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 3 — Community Structure

Schapiro (2017) §3.b: 15 items in 5 communities × 3 items each.

The community graph has a ring topology: communities form a cycle, and the boundary item of each community connects to the first item of the next community. This produces an asymmetry between **internal** items (connected only within their own community) and **boundary** items (connected to an adjacent community).

**Why this matters:**
- MSP accumulates statistical regularities: items that often co-occur (same community) develop similar CA1 representations. The 15×15 RSA matrix should show 5 warm blocks on the diagonal.
- TSP binds each individual episode: CA3 forms distinct attractors, preserving episode-specific detail.

**Metrics:**
- RSA heatmaps: 15×15 Pearson r (initial vs settled; DG/CA3/CA1)
- Output probability: internal (degree 2) vs boundary (degree 3) items across epochs
  - Boundary items have extra between-community edges → slightly harder to predict correctly
- Settled-minus-initial CA1 heatmap: which items' representations change most through settling?
- Pattern similarity bars: within-community vs between-community (analogous to Fig 2b)

**Expected results (Schapiro 2017 Fig. 3):**
- RSA matrix shows 5 diagonal blocks (community clusters) after training
- Internal items reach slightly higher output probability (more predictable neighbors)
- Settling (20→80 cycles) improves output probability for all items (CA3 pattern completion)

In [ ]:
# --- Fig 3: Community structure training + testing ---
# 15 items, 5 communities × 3 items (Schapiro 2017 Fig. 1)
# 60 trials/epoch × 10 epochs (Schapiro 2017 §3.b)

N_ITEMS_COMM  = 15
N_COMM        = 5
IPC           = 3
N_EPOCHS_COMM = 10
N_TRIALS_COMM = 60

# Template instance for structural methods (rsa_masks, output_prob_by_epoch, internal/boundary_items)
ds_comm = T_CommunityGraphDataset(n_steps=N_TRIALS_COMM, n_communities=N_COMM, items_per_community=IPC)

# eval_fn returns (snap20, snap80): activity snapshots at cycle 20 and 80 per item.
# result['eval'][r][0]      = (snap20, snap80) BEFORE training  → snap20 for initial RSA
# result['eval'][r][ep+1]   = (snap20, snap80) after epoch ep   → snap80['ecout'] for output prob
# result['eval'][r][-1]     = (snap20, snap80) after final epoch → snap80 for settled RSA

print('Running community structure experiment...')
comm_results = run_simulation(
    lambda: M_Hip(n_cond=N_ITEMS_COMM),
    lambda: T_CommunityGraphDataset(
        n_steps=N_TRIALS_COMM, n_communities=N_COMM, items_per_community=IPC,
    ),
    n_epochs=N_EPOCHS_COMM, n_reps=N_REPS,
    eval_fn=lambda m: m.run_evaluation_all(),
    seed=0,
)

# Initial RSA: snap20 from pre-training eval (Schapiro 2017 Fig. 3, "before training")
comm_init_rsa = F_extract_rsa(comm_results, lambda e: e[0][0])
# Settled RSA: snap80 from post-final-epoch eval
comm_sett_rsa = F_extract_rsa(comm_results, lambda e: e[-1][1])

# ECout activity per rep per epoch (settled, 80 cycles)
comm_ecout = np.stack([
    np.stack([comm_results['eval'][r][ep + 1][1]['ecout'] for ep in range(N_EPOCHS_COMM)])
    for r in range(N_REPS)
])  # (N_REPS, N_EPOCHS_COMM, N_ITEMS_COMM, N_ITEMS_COMM)
# CA1 for settled-minus-initial difference plot
comm_sett_ca1 = np.stack([comm_results['eval'][r][-1][1]['ca1'] for r in range(N_REPS)])
comm_init_ca1 = np.stack([comm_results['eval'][r][0][0]['ca1']  for r in range(N_REPS)])

print('Done.')

In [ ]:
# --- Fig 3 metrics ---
epochs     = np.arange(1, N_EPOCHS_COMM + 1)
comm_prob  = ds_comm.output_prob_by_epoch(comm_ecout)   # {'internal': ..., 'boundary': ...}
comm_masks = ds_comm.rsa_masks()                        # {'within': ..., 'between': ...}
comm_sim   = F_pattern_sim_by_mask(comm_sett_rsa, comm_masks)

ca1_diff = comm_sett_ca1.mean(axis=0) - comm_init_ca1.mean(axis=0)
diff_mag  = np.abs(ca1_diff).mean(axis=1)

print(f"CA1 settled — within-community:   {comm_sim['ca1']['within'][0]:.3f} ± {comm_sim['ca1']['within'][1]:.3f}")
print(f"CA1 settled — between-community:  {comm_sim['ca1']['between'][0]:.3f} ± {comm_sim['ca1']['between'][1]:.3f}")

In [ ]:
# --- Plot Figure 3 ---
fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 6, figure=fig, hspace=0.5, wspace=0.5)
layers       = ['dg', 'ca3', 'ca1']
internal_set = set(ds_comm.internal_items)

for col, layer in enumerate(layers):
    plot_rsa_heatmap(fig.add_subplot(gs[0, col]), comm_init_rsa[layer].mean(0),
                     N_COMM, IPC, title=f'Initial\n{layer.upper()}',
                     ylabel='RSA (before training)' if col == 0 else None)
    plot_rsa_heatmap(fig.add_subplot(gs[1, col]), comm_sett_rsa[layer].mean(0),
                     N_COMM, IPC, title=f'Settled\n{layer.upper()}',
                     ylabel='RSA (after 10 epochs)' if col == 0 else None)

plot_output_prob(fig.add_subplot(gs[0, 3:5]), epochs, comm_prob,
                 chance=2 / N_ITEMS_COMM,
                 title='Output probability\n(internal vs boundary)')

ax = fig.add_subplot(gs[0, 5])
ax.bar(range(N_ITEMS_COMM), diff_mag,
       color=['C0' if i in internal_set else 'C1' for i in range(N_ITEMS_COMM)], alpha=0.8)
ax.set_xlabel('Item', fontsize=8)
ax.set_ylabel('|Δ CA1|', fontsize=8)
ax.set_title('CA1 settled−initial\nmagnitude', fontsize=9)
ax.set_xticks([])

plot_sim_bars(fig.add_subplot(gs[1, 3:]), comm_sim, layers,
              ['within', 'between'], title='Pattern similarity (settled)')

fig.suptitle('Figure 3 — Community Structure (Schapiro 2017)', fontsize=11, fontweight='bold')
plt.savefig(DIR_VIZ / 'Schapiro2017_Fig3_community.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 4 — Associative Inference

Schapiro (2017) §3.c: 9 items in 3 triads (ABC/DEF/GHI).

The model is trained only on **direct pairs**: A→B and B→C for each triad. The test asks whether it can complete **transitive pairs**: A→C (two hops). This requires inference — the model never sees A and C together during training.

**Why CA3 is needed for transitivity:**
- Without CA3 recurrence: the model stores only direct associations (A→B and B→C). When A is presented at test, ECout can activate B but cannot then use B to activate C.
- With CA3 recurrence: after training, A's CA3 attractor (TSP) encodes A's context including its connection to B. Through recurrent settling, CA3 can bridge the two-hop A→B→C path, producing CA1 patterns that include C-like components.

**Two conditions:**
- **Full model**: normal M_Hip with CA3→CA1 connection
- **No CA3 (lesion)**: CA3→CA1 weights zeroed; only DG→CA1 via MSP (W_ECin→CA1)

**Metrics:**
- RSA heatmaps: 9×9 Pearson r showing whether A and C cluster together (transitive inference)
- Pattern similarity: direct (A↔B, B↔C) vs transitive (A↔C) vs unrelated
- Output probability: P(ECout_C > 0.5 | input=A) for full vs lesion model

In [ ]:
# --- Chain (associative inference) task constants ---
N_ITEMS_CHAIN  = 9
N_TRIADS       = 3
IPC_CHAIN      = 3
N_EPOCHS_CHAIN = 10
N_TRIALS_CHAIN = 60

# Template instance for structural methods (rsa_masks, output_prob, direct_pairs, trans_pairs)
ds_chain = T_ChainDataset(n_steps=N_TRIALS_CHAIN, n_triads=N_TRIADS)

print('Direct pairs (trained):', ds_chain.direct_pairs)
print('Transitive pairs (test):', ds_chain.trans_pairs)
print(f'n_items={ds_chain.n_items}')

In [ ]:
# --- Fig 4: Associative inference training + testing ---
# Two conditions: full model (CA3 recurrence) vs CA3 lesion (W_CA3 zeroed).
# Lesion: init_fn zeros W_CA3 at model creation;
#         post_update_fn re-zeros W_CA3 after each CHL weight update,
#         so the CA3→CA1 projection never contributes to CA1 activity.

print('Running full model (with CA3 recurrence)...')
chain_full_results = run_simulation(
    lambda: M_Hip(n_cond=N_ITEMS_CHAIN),
    lambda: T_ChainDataset(n_steps=N_TRIALS_CHAIN),
    n_epochs=N_EPOCHS_CHAIN, n_reps=N_REPS,
    eval_fn=lambda m: m.run_evaluation_all()[1],
    seed=0,
)

print('Running lesion model (CA3→CA1 zeroed)...')
chain_les_results = run_simulation(
    lambda: M_Hip(n_cond=N_ITEMS_CHAIN),
    lambda: T_ChainDataset(n_steps=N_TRIALS_CHAIN),
    n_epochs=N_EPOCHS_CHAIN, n_reps=N_REPS,
    eval_fn=lambda m: m.run_evaluation_all()[1],
    init_fn=lambda m: m.ca1.W_CA3.data.zero_(),
    post_update_fn=lambda m: m.ca1.W_CA3.data.zero_(),
    seed=0,
)

# Final settled ECout and RSA matrices (last epoch eval, index -1)
chain_full_ecout = np.stack([chain_full_results['eval'][r][-1]['ecout'] for r in range(N_REPS)])
chain_les_ecout  = np.stack([chain_les_results['eval'][r][-1]['ecout']  for r in range(N_REPS)])
chain_full_rsa = F_extract_rsa(chain_full_results, lambda e: e[-1])
chain_les_rsa  = F_extract_rsa(chain_les_results,  lambda e: e[-1])

print('Done.')

In [ ]:
# --- Fig 4 metrics ---
chain_masks = ds_chain.rsa_masks()   # {'direct': ..., 'transitive': ..., 'unrelated': ...}
full_sim    = F_pattern_sim_by_mask(chain_full_rsa, chain_masks)
les_sim     = F_pattern_sim_by_mask(chain_les_rsa,  chain_masks)
full_prob   = ds_chain.output_prob(chain_full_ecout)
les_prob    = ds_chain.output_prob(chain_les_ecout)

print('Output probability (direct / transitive):')
print(f"  Full model: direct={full_prob['direct']:.3f}  transitive={full_prob['transitive']:.3f}")
print(f"  Lesion:     direct={les_prob['direct']:.3f}  transitive={les_prob['transitive']:.3f}")
print()
print('Pattern similarity summary (CA1):')
for cond, sim in [('Full', full_sim), ('Lesion', les_sim)]:
    d = sim['ca1']['direct'][0]; t = sim['ca1']['transitive'][0]; u = sim['ca1']['unrelated'][0]
    print(f'  {cond}: direct={d:.3f}  transitive={t:.3f}  unrelated={u:.3f}')

In [ ]:
# --- Plot Figure 4 ---
fig = plt.figure(figsize=(16, 6))
gs  = gridspec.GridSpec(1, 8, figure=fig, hspace=0.4, wspace=0.55)
layers     = ['dg', 'ca3', 'ca1']
conditions = [('Full model',                chain_full_rsa, full_sim, full_prob),
              ('CA3 lesion (no recurrence)', chain_les_rsa,  les_sim, les_prob)]

for cond_col, (cond_name, rsa, sim, prob) in enumerate(conditions):
    offset = cond_col * 4

    ax = fig.add_subplot(gs[0, offset])
    im = plot_rsa_heatmap(ax, rsa['ca1'].mean(0), N_TRIADS, IPC_CHAIN,
                          title=f'{cond_name}\nCA1 RSA (settled)')
    for k in range(N_TRIADS):
        ax.text(k * IPC_CHAIN + 1, -0.8, f'T{k}', ha='center', fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plot_sim_bars(fig.add_subplot(gs[0, offset + 1:offset + 3]), sim, layers,
                  ['direct', 'transitive', 'unrelated'], colors=['C0', 'C2', 'C1'])

    ax = fig.add_subplot(gs[0, offset + 3])
    ax.bar([0, 1], [prob['direct'], prob['transitive']], color=['C0', 'C2'], alpha=0.8, width=0.5)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['direct', 'transitive'], fontsize=8)
    ax.set_ylabel('P(output > 0.5)', fontsize=8)
    ax.set_title('Output probability', fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.axhline(2 / N_ITEMS_CHAIN, color='gray', lw=0.8, ls='--')

fig.suptitle('Figure 4 — Associative Inference (Schapiro 2017)', fontsize=11, fontweight='bold')
plt.savefig(DIR_VIZ / 'Schapiro2017_Fig4_inference.png', dpi=150, bbox_inches='tight')
plt.show()